<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 4 (Jueves) — No supervisado — Agrupación (K-means) y asociación (Apriori)**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🧺 Reglas de asociación: teoría de Apriori, soporte, confianza y lift, y Apriori desde cero

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Descubrir qué productos se compran juntos (sin variable objetivo). |
| 🔍 Comprensión de los datos | Canastas como matriz verdadero/falso. |
| 🧹 Preparación de los datos | Apriori desde cero con poda por subconjuntos infrecuentes. |
| 🤖 Modelado | Reglas con soporte, confianza y lift. |
| 📊 Evaluación | Efecto del soporte mínimo en el número de conjuntos y el tiempo. |
| 🚀 Despliegue y conclusiones | Buenas prácticas y límites. |

> 🎯 **Analogía:** un cajero que, de ver miles de tickets, se da cuenta de que "quien lleva pan y leche suele llevar huevos", y no necesita revisar todas las combinaciones posibles porque descarta de antemano las que ya son raras.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 👉 **4** | Jueves | **No supervisado: agrupación y asociación** |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

## 1. Entendimiento del problema (fase 1 de CRISP-DM)

Un comercio quiere saber **qué productos se compran juntos** para ubicarlos mejor, armar promociones y recomendar. Es un problema de **reglas de asociación**: no hay variable objetivo, solo transacciones (canastas) [3].

### 🧪 Teoría rápida: reglas de asociación

Una **regla de asociación** tiene la forma **A → B** ("quien compra A suele comprar también B") y se descubre en datos de **transacciones** (canastas de compra) [3]. Se evalúa con tres medidas:

| Medida | Fórmula | Se lee como |
|---|---|---|
| **Soporte** de A | $\text{sop}(A)=\dfrac{\#\{\text{transacciones que contienen }A\}}{\#\{\text{transacciones}\}}$ | qué tan frecuente es el conjunto |
| **Confianza** de A → B | $\text{conf}(A\to B)=\dfrac{\text{sop}(A\cup B)}{\text{sop}(A)}$ | probabilidad de B dado A |
| **Elevación** (*lift*) | $\text{lift}(A\to B)=\dfrac{\text{conf}(A\to B)}{\text{sop}(B)}$ | cuántas veces más probable es B si hay A, frente a B sin saber nada |

Un *lift* > 1 indica asociación positiva, = 1 independencia y < 1 que se **sustituyen** [5]. Otras medidas útiles son el **apalancamiento** (*leverage*) $\text{sop}(A\cup B)-\text{sop}(A)\,\text{sop}(B)$ y la **convicción** $\dfrac{1-\text{sop}(B)}{1-\text{conf}(A\to B)}$ [9].

**Algoritmo Apriori.** Se apoya en una propiedad: *todo subconjunto de un conjunto frecuente también es frecuente* (y, al revés, si un conjunto no es frecuente, ninguno de sus superconjuntos lo es), lo que permite **podar** candidatos [4]. El proceso tiene dos pasos: (1) hallar los **conjuntos frecuentes** con soporte ≥ `min_support`; (2) generar las **reglas** con confianza (o lift) ≥ umbral [8]. Alternativas más rápidas son **FP-Growth** [6] y **Eclat** [7].

## 2. Entendimiento de los datos

Una transacción es un conjunto de productos. Se representa como una **matriz de verdadero/falso** (filas = transacciones, columnas = productos). Usaremos 6 canastas pequeñas para poder verificar todo a mano.

In [ ]:
import itertools, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

canastas = [['pan', 'leche', 'huevos'], ['pan', 'leche'], ['pan', 'cafe'], ['leche', 'huevos'],
            ['pan', 'leche', 'huevos', 'cafe'], ['leche', 'cafe']]
te = TransactionEncoder()
M = pd.DataFrame(te.fit_transform(canastas), columns=te.columns_)
M

## 3–4. Modelado: Apriori desde cero

Principio de Apriori: *si un conjunto es infrecuente, todos sus superconjuntos también* [4]. Por eso en cada nivel solo se combinan los conjuntos frecuentes del nivel anterior. Se programa y se compara con `mlxtend` [10].

In [ ]:
def apriori_desde_cero(canastas, min_support):
    n = len(canastas)
    sop = lambda s: sum(set(s) <= set(t) for t in canastas) / n
    items = sorted({p for t in canastas for p in t})
    nivel = [frozenset([p]) for p in items if sop([p]) >= min_support]
    frecuentes = {s: sop(s) for s in nivel}
    k = 2
    while nivel:
        candidatos = {a | b for a in nivel for b in nivel if len(a | b) == k}
        # poda: todos los subconjuntos de tamaño k-1 deben ser frecuentes
        candidatos = [c for c in candidatos if all(frozenset(s) in frecuentes for s in itertools.combinations(c, k - 1))]
        nivel = [c for c in candidatos if sop(c) >= min_support]
        frecuentes.update({c: sop(c) for c in nivel})
        k += 1
    return frecuentes

mio = apriori_desde_cero(canastas, 0.33)
lib = apriori(M, min_support=0.33, use_colnames=True)
print('Conjuntos frecuentes (desde cero):', len(mio), '| (mlxtend):', len(lib))
print('¿Mismos conjuntos y soportes?', all(abs(mio[s] - v) < 1e-9 for s, v in zip(lib.itemsets, lib.support)) and len(mio) == len(lib))

# Reglas: para cada conjunto frecuente y cada partición antecedente -> consecuente
reglas = []
for conj, s in mio.items():
    for r in range(1, len(conj)):
        for ant in itertools.combinations(conj, r):
            ant = frozenset(ant); cons = conj - ant
            conf = s / mio[ant]; lift = conf / mio[cons]
            reglas.append((' + '.join(sorted(ant)), ' + '.join(sorted(cons)), round(s, 3), round(conf, 3), round(lift, 3)))
pd.DataFrame(reglas, columns=['antecedente', 'consecuente', 'soporte', 'confianza', 'lift']).sort_values('lift', ascending=False).head(8)

## 5. Evaluación: el efecto de `min_support`

Bajar el soporte mínimo hace crecer los conjuntos frecuentes (y el tiempo) de forma muy rápida. Lo medimos con 3 000 canastas sintéticas de 40 productos con popularidad desigual.

In [ ]:
rng = np.random.default_rng(0)
productos = [f'p{i}' for i in range(40)]
prob = np.array([1 / (i + 2) for i in range(40)]); prob /= prob.sum()
sint = [list(rng.choice(productos, size=rng.integers(2, 9), replace=False, p=prob)) for _ in range(3000)]
Ms = pd.DataFrame(TransactionEncoder().fit(sint).transform(sint), columns=sorted(set(sum(sint, []))))

filas = []
for s in [0.2, 0.1, 0.05, 0.02, 0.01]:
    t0 = time.perf_counter(); f = apriori(Ms, min_support=s, use_colnames=True); ms = (time.perf_counter() - t0) * 1000
    filas.append((s, len(f), round(ms, 1)))
res = pd.DataFrame(filas, columns=['min_support', 'conjuntos_frecuentes', 'tiempo_ms'])
print(res.to_string(index=False))
plt.semilogy(res.min_support, res.conjuntos_frecuentes, 'o-'); plt.xlabel('Soporte mínimo'); plt.ylabel('Conjuntos frecuentes (log)'); plt.gca().invert_xaxis(); plt.show()

## 6. Despliegue y buenas prácticas

- Elija `min_support` según la frecuencia mínima que justifique actuar; ni tan bajo que genere miles de reglas ni tan alto que oculte pares interesantes.
- Filtre por **lift > 1** (para no confundir asociación con popularidad) y por confianza [5].
- Una regla describe **co-ocurrencia, no causalidad** [9]; valide con un experimento (prueba A/B) antes de cambiar la tienda.
- Para muchos datos, use **FP-Growth** [6] o **Eclat** [7]; devuelven los mismos conjuntos que Apriori.

### ✍️ Ejercicios
1. Calcule a mano el soporte, la confianza y el lift de `cafe → leche` en las 6 canastas y compare con el código.
2. Con las 3 000 canastas sintéticas, ¿qué `min_support` deja menos de 100 conjuntos frecuentes?
3. Sustituya `apriori` por `fpgrowth` (de `mlxtend.frequent_patterns`) y verifique que obtiene los mismos conjuntos.

---
## ✅ Conclusiones y siguientes pasos

Apriori reduce el trabajo combinatorio con una idea simple: un conjunto con un subconjunto infrecuente no puede ser frecuente [4]. Las medidas de soporte, confianza y lift permiten filtrar, pero el soporte mínimo manda sobre el costo, y toda regla es asociación, no causalidad [9].

---
## 📚 Referencias

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Agrawal, R., Imieliński, T., & Swami, A. (1993). Mining association rules between sets of items in large databases. *Proceedings of the 1993 ACM SIGMOD International Conference on Management of Data*, 207–216. https://doi.org/10.1145/170035.170072

**[4]** Agrawal, R., & Srikant, R. (1994). Fast algorithms for mining association rules. *Proceedings of the 20th International Conference on Very Large Data Bases (VLDB)*, 487–499.

**[5]** Brin, S., Motwani, R., Ullman, J. D., & Tsur, S. (1997). Dynamic itemset counting and implication rules for market basket data. *Proceedings of the 1997 ACM SIGMOD International Conference on Management of Data*, 255–264.

**[6]** Han, J., Pei, J., & Yin, Y. (2000). Mining frequent patterns without candidate generation. *Proceedings of the 2000 ACM SIGMOD International Conference on Management of Data*, 1–12.

**[7]** Zaki, M. J. (2000). Scalable algorithms for association mining. *IEEE Transactions on Knowledge and Data Engineering, 12*(3), 372–390. https://doi.org/10.1109/69.846291

**[8]** Han, J., Kamber, M., & Pei, J. (2012). *Data Mining: Concepts and Techniques* (3.ª ed.). Morgan Kaufmann. https://doi.org/10.1016/C2009-0-61819-5

**[9]** Tan, P.-N., Steinbach, M., Karpatne, A., & Kumar, V. (2019). *Introduction to Data Mining* (2.ª ed.). Pearson.

**[10]** Raschka, S. (2018). MLxtend: Providing machine learning and data science utilities and extensions to Python's scientific computing stack. *Journal of Open Source Software, 3*(24), 638. https://doi.org/10.21105/joss.00638